# Ch 8. Tool Calling 기초

원본: [AI Assistant Engineering - Part 2 Ch 8](https://desty.github.io/study-ai-assistant-engineering/part2/08-tool-calling/)

## 이 챕터에서 배우는 것

- **Tool Calling (Function Calling)** — LLM이 우리 코드의 함수를 부르게 하는 방식
- **툴 3종류** (Data / Action / Orchestration) — OpenAI Practical Guide 분류
- LLM과 툴 사이의 **tool_use ↔ tool_result 루프** 구조
- **Pydantic 으로 파라미터 검증** + **승인 기반 실행** (부작용 있는 툴)
- 무한 루프·잘못된 파라미터·부작용 처리 — 실전에서 망하는 포인트들
- Part 5 Agent 로 가는 다리

In [ ]:
# 필수 패키지
!pip install -q anthropic pydantic

In [ ]:
# API 키 설정
import os
from getpass import getpass

if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')

In [ ]:
from anthropic import Anthropic
from pydantic import BaseModel, Field, ValidationError
import json

client = Anthropic()

## 4. 최소 예제 — 계산기 툴

In [ ]:
tools = [
    {
        "name": "calculate",
        "description": "간단한 산술 계산을 수행. 복잡한 금액·환율·수량 계산에 사용.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "파이썬이 평가 가능한 산술식. 예: '1000 * 1.08 / 12'"
                },
            },
            "required": ["expression"],
        },
    }
]

def run_calculate(expression: str) -> str:  # (1)!
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

# 1) 첫 호출
messages = [{"role": "user", "content": "월 1,000달러 저금을 연 5%로 3년 하면 이자만 얼마?"}]
r = client.messages.create(
    model="claude-haiku-4-5-20251001", max_tokens=1024,
    tools=tools, messages=messages,
)

# 2) tool_use 응답을 받았나?
if r.stop_reason == "tool_use":
    for block in r.content:
        if block.type == "tool_use":
            print(f"LLM 요청 툴: {block.name}, 인수: {block.input}")
            tool_result = run_calculate(**block.input)  # (2)!
            print(f"계산 결과: {tool_result}")

            # 3) tool_result 포함해 다시 호출
            messages.append({"role": "assistant", "content": r.content})
            messages.append({
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": tool_result,
                }],
            })
            r2 = client.messages.create(
                model="claude-haiku-4-5-20251001", max_tokens=1024,
                tools=tools, messages=messages,
            )
            print(f"\n최종 답변: {r2.content[0].text}")

## 5. 실전 튜토리얼

### 5.1 여러 툴 · Pydantic 검증

In [ ]:
class WeatherArgs(BaseModel):
    city: str = Field(..., min_length=1)
    units: str = Field(default="metric", pattern=r"^(metric|imperial)$")

class OrderLookupArgs(BaseModel):
    order_id: str = Field(..., pattern=r"^[A-Z]-\d+$")

TOOL_SPECS = {
    "get_weather": {
        "schema": WeatherArgs,
        "tool": {
            "name": "get_weather",
            "description": "도시의 현재 날씨 조회",
            "input_schema": WeatherArgs.model_json_schema(),  # (1)!
        },
        "handler": lambda args: f"{args.city} 날씨 15도 맑음",
    },
    "lookup_order": {
        "schema": OrderLookupArgs,
        "tool": {
            "name": "lookup_order",
            "description": "주문번호로 주문 조회 (형식: A-123)",
            "input_schema": OrderLookupArgs.model_json_schema(),
        },
        "handler": lambda args: f"{args.order_id}: 배송중, ETA 2일",
    },
}

def dispatch(tool_name: str, raw_input: dict) -> str:
    spec = TOOL_SPECS[tool_name]
    try:
        args = spec["schema"].model_validate(raw_input)  # (2)!
    except ValidationError as e:
        return f"invalid_input: {e}"
    return spec["handler"](args)

# 테스트
result = dispatch("get_weather", {"city": "서울"})
print(f"dispatch 결과: {result}")

### 5.2 루프가 **여러 번** 도는 경우

In [ ]:
def chat_with_tools(user_msg: str, max_steps: int = 5) -> str:
    messages = [{"role": "user", "content": user_msg}]
    tools = [s["tool"] for s in TOOL_SPECS.values()]

    for step in range(max_steps):  # (1)!
        r = client.messages.create(
            model="claude-haiku-4-5-20251001", max_tokens=1024,
            tools=tools, messages=messages,
        )
        messages.append({"role": "assistant", "content": r.content})

        if r.stop_reason != "tool_use":  # 최종 답변
            return "".join(b.text for b in r.content if hasattr(b, "text"))

        # tool_use 블록들 처리
        tool_results = []
        for block in r.content:
            if block.type == "tool_use":
                result = dispatch(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })
        messages.append({"role": "user", "content": tool_results})

    return "[max_steps 초과]"

# 테스트
result = chat_with_tools("서울 날씨 어때?")
print(f"결과: {result}")

### 5.3 승인 기반 실행 (Action 툴)

In [ ]:
RISKY_TOOLS = {"send_email", "cancel_order", "charge_card"}

def dispatch_with_approval(tool_name: str, args: dict) -> str:
    if tool_name in RISKY_TOOLS:
        print(f"경고: {tool_name}({args}) 승인 요청")
        if input("y/N > ").strip().lower() != "y":
            return "declined_by_user"
    return dispatch(tool_name, args)

print("승인 기반 실행 함수가 정의되었습니다.")
print("실전에선: Slack 버튼, 대시보드 승인 큐, LangGraph interrupt 등 사용")

### 5.4 에러·타임아웃 처리

In [ ]:
import requests

def get_weather_safe(city: str) -> str:
    try:
        r = requests.get(f"https://api.weather.com/{city}", timeout=5)
        return r.json()["description"]
    except requests.Timeout:
        return "error: weather API timeout"  # (1)!
    except Exception as e:
        return f"error: {e}"

print("에러·타임아웃 처리 패턴 정의")
print("에러 문자열을 tool_result로 반환하면 LLM이 자연어로 복구 시도")

## 6. 자주 깨지는 포인트

### 실수 1. `eval` 을 그대로 씀
- `eval(expression)` 은 임의 코드 실행 위험
- 대응: `asteval` · `numexpr` 같은 안전 평가 라이브러리 사용

### 실수 2. 무한 루프
- LLM이 같은 툴을 계속 호출
- `max_steps` 없으면 토큰·비용 폭주
- 대응: 루프 상한 5~10회. 초과 시 명시적 에러

### 실수 3. 파라미터 검증 누락
- LLM이 잘못된 타입 (`quantity: "two"`) 넘기기
- 대응: **Pydantic** 으로 무조건 검증

### 실수 4. Action 툴을 승인 없이 실행
- 결제·삭제·발송이 모델 판단만으로 실행
- 대응: §5.3. `RISKY_TOOLS` 목록 + 승인 큐

### 실수 5. 툴 이름·설명이 모호
- `query_data`, `do_thing` → LLM 선택 실패
- 대응: 동사 + 명확한 명사. description에 언제 쓰고 언제 안 쓰는지 명시

### 실수 6. tool_result 를 빠뜨리고 다음 호출
- `tool_use` 받고 바로 새 user 메시지 보내기
- 대응: 루프 구조를 함수로 감싸 실수 방지

## 7. 운영 시 체크할 점

- [ ] **루프 상한** `max_steps` 5~10
- [ ] **툴 입력 전부 Pydantic 검증** · 실패 시 `invalid_input` 반환
- [ ] **Action 툴 화이트리스트** + 승인 큐 + 감사 로그
- [ ] **실행 결과에 크기 제한** — 큰 쿼리 결과를 그대로 넣으면 컨텍스트 초과
- [ ] **툴별 타임아웃** — Data 5초, Action 30초 등
- [ ] **관측성** — 어떤 툴을 · 몇 번 · 얼마나 걸려서 실행했는지 로그 (LangSmith/Langfuse)
- [ ] **비용 · 지연 모니터링** — 툴 루프가 길면 비용 빠르게 증가
- [ ] **샌드박스** — 코드 실행형 툴은 Docker 등 격리 환경

## Part 2를 마치며

Part 2에서 배운 것 (5 챕터):

| Ch | 무엇을 | 실전 의미 |
|---|---|---|
| 4 | API 호출 · 에러·재시도 | 모든 LLM 앱의 시작점 |
| 5 | 프롬프트 · Few-shot · CoT | 모델의 행동을 "계약"으로 고정 |
| 6 | 구조화 출력 (Pydantic · tool-use schema) | 파이프라인 다음 단계가 쓸 수 있는 JSON |
| 7 | 스트리밍 · UX | 사용자 체감 속도 |
| 8 | **Tool Calling** | LLM에게 손을 달아주기 — Agent의 시작 |

**Part 2 졸업 상태** — 여기까지면 다음 중 하나를 **PoC 수준으로** 만들 수 있어야 합니다:

- 고객 문의 자동 분류 + 간단 응답 (구조화 출력 기반)
- 문서 + 툴 조회 봇 (Tool Calling + 간단 RAG 예고)
- 스트리밍 챗봇 웹 UI (FastAPI + SSE)

---

**다음 Part** → Part 3. RAG — 외부 지식을 붙이는 법

여기까지는 "LLM이 학습한 것"만 활용했습니다. 이제 **우리 회사의 문서·DB**를 붙여 답변의 근거를 확보합니다.